### Build End-to-End Pipeline Dataset: 
### Orders data with: 1)NULL values 2)String salary 3)Date column 
### Tasks: 1)Clean NULLs 2)Cast columns 3)Add derived columns 4)Apply UDF 
### Load: 1)Full load 2)Incremental load 3)Parameterize notebook

### 1. SAMPLE DATASET (Dirty + Realistic)
### 1)NULLs 2)String salary 3)Dates 4)Updates (for incremental load)
### from pyspark.sql import SparkSession

### spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()

### data = [
###     (1, "C001", "Laptop", "50000", "2024-01-01"),
###    (2, "C002", "Mobile", None, "2024-01-02"),
###    (3, "C003", "Tablet", "20000", "2024-01-03"),
###    (4, "C004", "Laptop", "55000", "2024-01-04"),
###    (5, "C005", "Headphones", None, "2024-01-05"),
###    ## Dont use this for first time (3, "C003", "Tablet", "22000", "2024-01-06"),  # updated
###    (6, "C006", "Camera", "30000", "2024-01-06"),
###    (7, "C007", "Mobile", "18000", "2024-01-07"),
###    (8, "C008", "Watch", "8000", "2024-01-07")
###]

### columns = ["order_id", "customer_id", "product", "amount", "updated_date"]

### df = spark.createDataFrame(data, columns)
### TASK LIST
### Task 1: Handle NULLs
### Replace NULL amount with 1000
### Task 2: Cast Columns
### Convert amount → integer
### Convert updated_date → date
### Task 3: Add Derived Columns
### bonus = amount * 0.1
### category:
### 20000 → High
### else → Low
### Task 4: Apply UDF
### Create amount_bucket:
### < 10000 → Low
### 10000–30000 → Medium
### 30000 → High
### Task 5: Full Load
### Load all data to target
### Task 6: Incremental Load
### Load only new/updated records
### Handle duplicates (keep latest)
### Task 7: Parameterization
### Pass:
### input path
last_loaded_date

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.appName("E2E_Pipeline").getOrCreate()
data = [
(1, "C001", "Laptop", "50000", "2024-01-01"),
(2, "C002", "Mobile", None, "2024-01-02"),
(3, "C003", "Tablet", "20000", "2024-01-03"),
(4, "C004", "Laptop", "55000", "2024-01-04"),
(5, "C005", "Headphones", None, "2024-01-05"),
(6, "C006", "Camera", "30000", "2024-01-06"),
(7, "C007", "Mobile", "18000", "2024-01-07"),
(8, "C008", "Watch", "8000", "2024-01-07")
]
df=spark.createDataFrame(data,['id',"product_id","product","amount","update_date"])
df.show()

+---+----------+----------+------+-----------+
| id|product_id|   product|amount|update_date|
+---+----------+----------+------+-----------+
|  1|      C001|    Laptop| 50000| 2024-01-01|
|  2|      C002|    Mobile|  NULL| 2024-01-02|
|  3|      C003|    Tablet| 20000| 2024-01-03|
|  4|      C004|    Laptop| 55000| 2024-01-04|
|  5|      C005|Headphones|  NULL| 2024-01-05|
|  6|      C006|    Camera| 30000| 2024-01-06|
|  7|      C007|    Mobile| 18000| 2024-01-07|
|  8|      C008|     Watch|  8000| 2024-01-07|
+---+----------+----------+------+-----------+



In [0]:
df.printSchema()

root
 |-- id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: string (nullable = true)
 |-- update_date: string (nullable = true)



In [0]:
df=df.withColumn('amount',df['amount'].cast('int'))

In [0]:
df_clean=df.fillna({
    'amount':1000
})
df_clean.show()

+---+----------+----------+------+-----------+
| id|product_id|   product|amount|update_date|
+---+----------+----------+------+-----------+
|  1|      C001|    Laptop| 50000| 2024-01-01|
|  2|      C002|    Mobile|  1000| 2024-01-02|
|  3|      C003|    Tablet| 20000| 2024-01-03|
|  4|      C004|    Laptop| 55000| 2024-01-04|
|  5|      C005|Headphones|  1000| 2024-01-05|
|  6|      C006|    Camera| 30000| 2024-01-06|
|  7|      C007|    Mobile| 18000| 2024-01-07|
|  8|      C008|     Watch|  8000| 2024-01-07|
+---+----------+----------+------+-----------+



In [0]:
df_clean=df_clean.withColumn("update_date",df_clean["update_date"].cast("date"))

In [0]:
df_clean.printSchema()

root
 |-- id: long (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = false)
 |-- update_date: date (nullable = true)



In [0]:
df_clean=df_clean.withColumn("bonus",df_clean["amount"]*0.1)
df_clean.show()

+---+----------+----------+------+-----------+------+
| id|product_id|   product|amount|update_date| bonus|
+---+----------+----------+------+-----------+------+
|  1|      C001|    Laptop| 50000| 2024-01-01|5000.0|
|  2|      C002|    Mobile|  1000| 2024-01-02| 100.0|
|  3|      C003|    Tablet| 20000| 2024-01-03|2000.0|
|  4|      C004|    Laptop| 55000| 2024-01-04|5500.0|
|  5|      C005|Headphones|  1000| 2024-01-05| 100.0|
|  6|      C006|    Camera| 30000| 2024-01-06|3000.0|
|  7|      C007|    Mobile| 18000| 2024-01-07|1800.0|
|  8|      C008|     Watch|  8000| 2024-01-07| 800.0|
+---+----------+----------+------+-----------+------+



In [0]:
df_clean=df_clean.withColumn("category", when(df_clean["amount"]>6000,"High").otherwise("Low"))
df_clean.show()

+---+----------+----------+------+-----------+------+--------+
| id|product_id|   product|amount|update_date| bonus|category|
+---+----------+----------+------+-----------+------+--------+
|  1|      C001|    Laptop| 50000| 2024-01-01|5000.0|    High|
|  2|      C002|    Mobile|  1000| 2024-01-02| 100.0|     Low|
|  3|      C003|    Tablet| 20000| 2024-01-03|2000.0|    High|
|  4|      C004|    Laptop| 55000| 2024-01-04|5500.0|    High|
|  5|      C005|Headphones|  1000| 2024-01-05| 100.0|     Low|
|  6|      C006|    Camera| 30000| 2024-01-06|3000.0|    High|
|  7|      C007|    Mobile| 18000| 2024-01-07|1800.0|    High|
|  8|      C008|     Watch|  8000| 2024-01-07| 800.0|    High|
+---+----------+----------+------+-----------+------+--------+



In [0]:
def grade(amount):
    if amount<10000:
        return "Low"
    elif amount>=10000 and amount<30000:
        return "Medium"
    elif amount>=30000:
        return "High"

In [0]:
from pyspark.sql.types import *
grade_udf=udf(grade,StringType())

In [0]:
df_clean=df_clean.withColumn("grade_udf",grade_udf(df_clean["amount"]))
df_clean.show()

+---+----------+----------+------+-----------+------+--------+---------+
| id|product_id|   product|amount|update_date| bonus|category|grade_udf|
+---+----------+----------+------+-----------+------+--------+---------+
|  1|      C001|    Laptop| 50000| 2024-01-01|5000.0|    High|     High|
|  2|      C002|    Mobile|  1000| 2024-01-02| 100.0|     Low|      Low|
|  3|      C003|    Tablet| 20000| 2024-01-03|2000.0|    High|   Medium|
|  4|      C004|    Laptop| 55000| 2024-01-04|5500.0|    High|     High|
|  5|      C005|Headphones|  1000| 2024-01-05| 100.0|     Low|      Low|
|  6|      C006|    Camera| 30000| 2024-01-06|3000.0|    High|     High|
|  7|      C007|    Mobile| 18000| 2024-01-07|1800.0|    High|   Medium|
|  8|      C008|     Watch|  8000| 2024-01-07| 800.0|    High|      Low|
+---+----------+----------+------+-----------+------+--------+---------+



In [0]:
df_clean.write \
    .mode("overwrite") \
    .parquet("/Volumes/demo/default/demo")

In [0]:
df_old = spark.read.parquet("/Volumes/demo/default/demo")
df_old.display()

id,product_id,product,amount,update_date,bonus,category,grade_udf
5,C005,Headphones,1000,2024-01-05,100.0,Low,Low
3,C003,Tablet,20000,2024-01-03,2000.0,High,Medium
7,C007,Mobile,18000,2024-01-07,1800.0,High,Medium
1,C001,Laptop,50000,2024-01-01,5000.0,High,High
4,C004,Laptop,55000,2024-01-04,5500.0,High,High
6,C006,Camera,30000,2024-01-06,3000.0,High,High
8,C008,Watch,8000,2024-01-07,800.0,High,Low
2,C002,Mobile,1000,2024-01-02,100.0,Low,Low


In [0]:
from pyspark.sql.functions import max

last_loaded_date = df_old.select(
    max("update_date")
).collect()[0][0]

print("Last Loaded Date:", last_loaded_date)

Last Loaded Date: 2024-01-07
